# Book Recommendation System

Рекомендательная система книг на основе датасета Book-Crossing.

Метрики сходства: Жаккар, Евклидово расстояние, авторская метрика.

In [1]:
import pandas as pd
import numpy as np

## 1. Загрузка и предобработка данных

In [2]:
# Загружаем оценки и книги (для названий)
books = pd.read_csv('Books.csv', low_memory=False)
ratings = pd.read_csv('Ratings.csv')

# Убираем неявные оценки (0 = просто взаимодействие без оценки)
ratings = ratings[ratings['Book-Rating'] > 0]

# Добавляем названия книг
df = ratings.merge(books[['ISBN', 'Book-Title']], on='ISBN')

In [3]:
# Фильтруем — оставляем книги с >= 20 оценками и пользователей с >= 5
book_counts = df['Book-Title'].value_counts()
user_counts = df['User-ID'].value_counts()

df = df[df['Book-Title'].isin(book_counts[book_counts >= 20].index)]
df = df[df['User-ID'].isin(user_counts[user_counts >= 5].index)]

In [4]:
# Матрица пользователь-книга
user_book_matrix = df.pivot_table(index='User-ID', columns='Book-Title', values='Book-Rating', fill_value=0)

# Нормализация оценок (Min-Max, от 0 до 1) для корректной работы евклидова расстояния
r_min = user_book_matrix[user_book_matrix > 0].min().min()
r_max = user_book_matrix.max().max()
user_book_norm = user_book_matrix.where(user_book_matrix == 0, (user_book_matrix - r_min) / (r_max - r_min))

## 2. Метрики сходства

In [5]:
# Жаккар — какая доля книг совпадает у двух пользователей
def jaccard_similarity(user1, user2):
    books1 = set(user1[user1 > 0].index)
    books2 = set(user2[user2 > 0].index)
    if len(books1) == 0 and len(books2) == 0:
        return 0.0
    return len(books1 & books2) / len(books1 | books2)

In [6]:
# Евклидово расстояние — насколько похоже оценивают общие книги
def euclidean_similarity(user1, user2):
    common = (user1 > 0) & (user2 > 0)
    if common.sum() == 0:
        return 0.0
    dist = np.sqrt(((user1[common] - user2[common]) ** 2).sum())
    return 1 / (1 + dist)

In [7]:
# Авторская метрика — сходство по профилю читателя
# Сравниваем КАК оценивают: средняя оценка, разброс, доля высоких оценок
def reading_profile_similarity(user1, user2):
    r1 = user1[user1 > 0]
    r2 = user2[user2 > 0]
    if len(r1) == 0 or len(r2) == 0:
        return 0.0

    mean_diff = abs(r1.mean() - r2.mean())
    std_diff = abs(r1.std() - r2.std())
    high_diff = abs((r1 >= 0.7).mean() - (r2 >= 0.7).mean())

    return 1 - (mean_diff + std_diff + high_diff) / 3

In [8]:
# Комбинированная метрика — взвешенная сумма трёх
def combined_similarity(user1, user2, w1=0.3, w2=0.4, w3=0.3):
    j = jaccard_similarity(user1, user2)
    e = euclidean_similarity(user1, user2)
    r = reading_profile_similarity(user1, user2)
    return w1 * j + w2 * e + w3 * r

## 3. Рекомендации

In [9]:
def get_recommendations(my_ratings, top_k=20, top_n=5):
    # Находим книги из ввода в датасете
    my_vector = pd.Series(0.0, index=user_book_norm.columns)
    matched_titles = set()
    for title, rating in my_ratings.items():
        matches = [b for b in my_vector.index if title.lower() in b.lower()]
        if matches:
            my_vector[matches[0]] = (rating - r_min) / (r_max - r_min)
            matched_titles.add(matches[0])
            print(f'  Найдено: "{matches[0]}" — оценка {rating}')
        else:
            print(f'  Не найдено: "{title}"')

    if len(matched_titles) == 0:
        return 'Ни одна книга не найдена в датасете'

    # Ищем пользователей с хотя бы 1 общей книгой
    similarities = {}
    for user_id in user_book_norm.index:
        user = user_book_norm.loc[user_id]
        if ((my_vector > 0) & (user > 0)).sum() == 0:
            continue
        similarities[user_id] = combined_similarity(my_vector, user)

    # Топ-K похожих
    similar_users = sorted(similarities, key=similarities.get, reverse=True)[:top_k]

    # Собираем книги которые они оценили >= 7, а мы не читали
    candidates = {}
    for user_id in similar_users:
        user = user_book_matrix.loc[user_id]
        sim = similarities[user_id]
        for book in user[user >= 7].index:
            if book not in matched_titles:
                if book not in candidates:
                    candidates[book] = 0
                candidates[book] += sim

    result = sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return pd.DataFrame([r[0] for r in result], columns=['Рекомендуемые книги'])

In [10]:
# Вводим книги которые читали и оценки (1-10)
# Названия должны быть из датасета (достаточно части названия)
my_ratings = {
    'Harry Potter': 9,
    'The Great Gatsby': 7,
    'The Hobbit': 8
}

get_recommendations(my_ratings)

  Найдено: "Harry Potter and the Chamber of Secrets (Book 2)" — оценка 9
  Найдено: "The GREAT GATSBY (Scribner Classic)" — оценка 7
  Найдено: "The Hobbit" — оценка 8


,Рекомендуемые книги
0,Harry Potter and the Prisoner of Azkaban (Book 3)
1,Harry Potter and the Goblet of Fire (Book 4)
2,Harry Potter and the Sorcerer's Stone (Book 1)
3,Harry Potter and the Order of the Phoenix (Boo...
4,Love in the Time of Cholera
